# **Spark Project - Create Table Notebook**

From Lecture 3_7:

**Part 1: Build a curated order-item analysis table**

In [0]:
orders = spark.read.parquet("/Volumes/workspace/default/project/project_files2/orders.parquet")
order_products = spark.read.parquet("/Volumes/workspace/default/project/project_files2/order_products.parquet")
products = spark.read.parquet("/Volumes/workspace/default/project/project_files2/products.parquet")
aisles = spark.read.parquet("/Volumes/workspace/default/project/project_files2/aisles.parquet")
departments = spark.read.parquet("/Volumes/workspace/default/project/project_files2/departments.parquet")

In [0]:
orders.show(2, truncate=False)
order_products.show(2, truncate=False)
products.show(2, truncate=False)
aisles.show(2, truncate=False)
departments.show(2, truncate=False)

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|2539329 |1      |prior   |1           |2        |8                |NULL                  |
|2398795 |1      |prior   |2           |3        |7                |15.0                  |
+--------+-------+--------+------------+---------+-----------------+----------------------+
only showing top 2 rows
+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|2       |33120     |1                |1        |
|2       |28985     |2                |1        |
+--------+----------+-----------------+---------+
only showing top 2 rows
+----------+--------------------------+--------+-------------+
|product_id|product_name             

In [0]:
order_products.join(orders, on="order_id", how="inner").show()

+--------+----------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|product_id|add_to_cart_order|reordered|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+----------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+
|     122|     17794|                1|        0| 127081|   prior|          16|        2|                1|                   5.0|
|     122|     18644|                2|        0| 127081|   prior|          16|        2|                1|                   5.0|
|     286|     10102|                1|        1|  91926|   prior|           2|        1|               12|                  30.0|
|     286|      5450|                2|        1|  91926|   prior|           2|        1|               12|                  30.0|
|     286|     11182|                3|        1|  91926|   prior|           2|    

In [0]:
morning_products = (
    order_products
    .join(orders, on="order_id", how="inner")
    .where("order_hour_of_day >= 8 AND order_hour_of_day <= 10")
    .selectExpr("order_id", "product_id", "order_hour_of_day as hour")
)

morning_products.show()

+--------+----------+----+
|order_id|product_id|hour|
+--------+----------+----+
|       2|     33120|   9|
|       2|     28985|   9|
|       2|      9327|   9|
|       2|     45918|   9|
|       2|     30035|   9|
|       2|     17794|   9|
|       2|     40141|   9|
|       2|      1819|   9|
|       2|     43668|   9|
|       4|     46842|   9|
|       4|     26434|   9|
|       4|     39758|   9|
|       4|     27761|   9|
|       4|     10054|   9|
|       4|     21351|   9|
|       4|     22598|   9|
|       4|     34862|   9|
|       4|     40285|   9|
|       4|     17616|   9|
|       4|     25146|   9|
+--------+----------+----+
only showing top 20 rows


**Beginning of Spark Project Content:**

In [0]:
order_details = (
    order_products
    .join(orders, on="order_id", how="inner") # USING AN INNER JOIN BECAUSE I ONLY CARE ABOUT MATCHING PAIRS
)

In [0]:
order_details = (
    order_products
    .join(orders, on="order_id", how="inner")
    .join(products, on="product_id", how="left") # USING A LEFT JOIN BECAUSE I WANT ALL OF THE ORDER PRODUCTS
)

In [0]:
order_details = (
    order_products
    .join(orders, on="order_id", how="inner")
    .join(products, on="product_id", how="left")
    .join(aisles, on="aisle_id", how="left")
    .join(departments, on="department_id", how="left")
)

In [0]:
order_details = order_details.select(
    "order_id",
    "user_id",
    "product_id",
    "product_name",
    "aisle",
    "department",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "add_to_cart_order",
    "reordered"
)

In [0]:
order_details.select(
    "product_name",
    "order_id",
    "aisle",
    "department"
).show(10)

+--------------------+--------+--------------------+----------+
|        product_name|order_id|               aisle|department|
+--------------------+--------+--------------------+----------+
|  Organic Egg Whites|       2|                eggs|dairy eggs|
|Michigan Organic ...|       2|    fresh vegetables|   produce|
|       Garlic Powder|       2|   spices seasonings|    pantry|
|      Coconut Butter|       2|       oils vinegars|    pantry|
|   Natural Sweetener|       2|  baking ingredients|    pantry|
|             Carrots|       2|    fresh vegetables|   produce|
|Original Unflavor...|       2|doughs gelatins b...|    pantry|
|All Natural No St...|       2|             spreads|    pantry|
|Classic Blend Col...|       2|packaged vegetabl...|   produce|
|Total 2% with Str...|       3|              yogurt|dairy eggs|
+--------------------+--------+--------------------+----------+
only showing top 10 rows


One row in order details represents a product from a particular order, and the metadata that goes along with that product, such as aisle and department. 

In [0]:
orders.count()

3421083

In [0]:
order_details.count()

33819106

THE ROW COUNT IS MUCH LARGER THAN ORDERS BECAUSE EACH ORDER CONTAINS MULTIPLE PRODUCTS, AND EACH PRODUCT THEN CREATES ITS OWN ROW. 

In [0]:
from pyspark.sql.functions import col, sum, when

total_rows = order_details.count()

null_counts = order_details.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in order_details.columns
])

null_counts.show(vertical=True, truncate=False)

-RECORD 0-------------------------
 order_id               | 0       
 user_id                | 0       
 product_id             | 0       
 product_name           | 18179   
 aisle                  | 152901  
 department             | 18179   
 order_number           | 0       
 order_dow              | 0       
 order_hour_of_day      | 0       
 days_since_prior_order | 2078068 
 add_to_cart_order      | 0       
 reordered              | 0       



THE ABOVE CODE COUNTS HOW MANY NULL VALUES THERE ARE IN EACH COLUMN OF THE DATAFRAME

THERE ARE NULLS ON:
- product_name
- aisle
- department

BECAUSE THESE WERE INCLUDED WITH A LEFT JOIN, AND ARE MISSING METADATA

THE FIELD "days_since_prior_order" HAS NULL VALUES BECAUSE THE FIRST ORDER FROM A CUSTOMER DOES NOT HAVE ANY PRIOR ORDERS.

IF ALL WERE INNER JOINS, THE ROWS WITH MISSING product_name, aisle, AND department INFORMATION WOULD HAVE BEEN DROPPED, AND I WOULD LOSE VAILD ORDER-PRODUCT ROWS. 

**Part 2: Persisting order_details**

In [0]:
order_details.write.mode("overwrite").saveAsTable("order_details_curated")